# Sample Question Generation

1. Extract metadata (categories, keywords, key ideas) from each seed question via GPT-5.
2. Randomly sample parts of that metadata.
3. Generate a new graduate-level problem grounded in the sampled metadata.

In [ ]:
import os, json, random
from pathlib import Path
import litellm

os.environ["OPENAI_API_KEY"] = ""  # paste key here

MODEL = "openai/gpt-5"
HERE = Path.cwd()

with open(HERE / "sampled_questions.json") as f:
    questions = json.load(f)

len(questions)

In [5]:
print(questions[0]['self_contained_problem'])

For an odd prime $p$ and an integer $a$ coprime to $p$, let $\left(\frac{a}{p}\right)$ denote the Legendre symbol. For nonnegative integers $k$, write the multinomial coefficient
$$\binom{4k}{k,k,k,k} = \frac{(4k)!}{(k!)^4}.$$
Prove or disprove the following three statements.

(a) For every prime $p > 3$ with $p \neq 11$,
$$\sum_{k=0}^{p-1} \frac{1}{15842^{k}}\binom{4k}{k,k,k,k} \equiv \begin{cases} 4x^{2} - 2p \pmod{p^{2}} & \text{if } \left(\tfrac{-11}{p}\right) = \left(\tfrac{2}{p}\right) = 1 \text{ and } p = x^{2} + 22y^{2}, \\ 2p - 8x^{2} \pmod{p^{2}} & \text{if } \left(\tfrac{-11}{p}\right) = \left(\tfrac{2}{p}\right) = -1 \text{ and } p = 2x^{2} + 11y^{2}, \\ 0 \pmod{p^{2}} & \text{if } \left(\tfrac{-11}{p}\right) = -\left(\tfrac{2}{p}\right). \end{cases}$$

(b) For every prime $p > 3$ with $p \neq 11$,
$$\sum_{k=0}^{p-1} \frac{280k + 19}{15842^{k}}\binom{4k}{k,k,k,k} \equiv 19\, p\,\left(\tfrac{-2}{p}\right) \pmod{p^{3}}.$$

(c) For every integer $n \geq 2$, the rational number

## Step 1 — Question → Metadata

In [6]:
META_PROMPT = """You are an expert mathematician. Read the problem and produce a compact metadata record describing it ABSTRACTLY enough that a fresh problem could be created from the metadata alone, WITHOUT copying or paraphrasing the original problem statement.

Return STRICT JSON with exactly this schema (no markdown, no commentary):
{
  "categories": [string],          // 2-5 MSC-style subject areas, fine-grained (e.g. "Algebraic Number Theory / Quadratic Reciprocity")
  "keywords": [string],            // 5-10 specific technical terms (objects, theorems, structures)
  "key_ideas": [string],           // 3-6 short noun-phrases naming the conceptual moves the problem hinges on
  "prerequisites": [string],       // 3-6 background topics a solver must know
  "techniques": [string],          // 2-5 proof / solution techniques typically applicable
  "difficulty": string             // one of: "undergraduate", "advanced-undergraduate", "graduate", "research"
}

Be SPECIFIC. Avoid generic words like "analysis" or "algebra" alone — qualify them.
Do NOT include any phrase that would let a reader reconstruct the original problem statement verbatim.

Problem:
{problem}
"""

def extract_metadata(problem: str) -> dict:
    resp = litellm.completion(
        model=MODEL,
        messages=[{"role": "user", "content": META_PROMPT.replace("{problem}", problem)}],
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

In [7]:
metadata_records = []
for q in questions:
    md = extract_metadata(q["self_contained_problem"])
    metadata_records.append({**q, "metadata": md})
    print(q["question_id"], "→", md.get("categories"))

with open(HERE / "metadata.json", "w") as f:
    json.dump(metadata_records, f, indent=2, ensure_ascii=False)

q_018 → ['p-adic Number Theory / Supercongruences for truncated hypergeometric series', 'Combinatorics / Hypergeometric identities and multinomial coefficients', 'Algebraic Number Theory / Binary quadratic forms and representations of primes', 'Modular Forms / CM forms and Hecke eigenvalues over quadratic fields']
q_036 → ['Computational Geometry / Visibility and Art Gallery Problems', 'Graph Theory / Geometric Graphs and Visibility Graphs', 'Combinatorial Optimization / Maximum Independent Set', 'Theoretical Computer Science / NP-Completeness and Reductions']
q_010 → ['Number Theory / Arithmetic progressions', 'Algebra / Systems of linear equations', 'Combinatorics / Finite sums and series']
q_017 → ['Nuclear Theory / Few-body systems and clustering', 'Quantum Many-Body Theory / Finite-temperature Green’s functions', 'Integral Equations in Quantum Mechanics / Faddeev–Yakubovsky formalism', 'Statistical Physics of Quantum Gases / Bose–Einstein condensation and Mott transitions']
q_064 

## Step 2 — Randomly sample parts of metadata

In [10]:
def sample_metadata(md: dict, seed: int | None = None) -> dict:
    rng = random.Random(seed)
    def pick(field, lo, hi):
        items = md.get(field, []) or []
        if not items:
            return []
        k = min(len(items), rng.randint(lo, hi))
        return rng.sample(items, k)
    return {
        "categories":    pick("categories",    1, 2),
        "keywords":      pick("keywords",      2, 4),
        "key_ideas":     pick("key_ideas",     1, 3),
        "prerequisites": pick("prerequisites", 1, 3),
        "techniques":    pick("techniques",    1, 2),
        "difficulty":    md.get("difficulty", "graduate"),
    }

sample_metadata(metadata_records[0]["metadata"], seed=0)

{'categories': ['Modular Forms / CM forms and Hecke eigenvalues over quadratic fields',
  'p-adic Number Theory / Supercongruences for truncated hypergeometric series'],
 'keywords': ['supercongruence modulo p^2 and p^3',
  'binary quadratic forms x^2 + Dy^2',
  'Gauss sums and Hasse–Davenport relation'],
 'key_ideas': ['evaluation through p-adic gamma expansions to second and third order',
  'case analysis controlled by two independent quadratic characters and their product'],
 'prerequisites': ['generalized hypergeometric series and Pochhammer notation',
  'binary quadratic forms and prime representation criteria',
  'p-adic valuations; Lucas and Kummer congruences; lifting-the-exponent'],
 'techniques': ['Greene–McCarthy finite field hypergeometric evaluation with Gross–Koblitz'],
 'difficulty': 'research'}

In [11]:
metadata_records[0]["metadata"]

{'categories': ['p-adic Number Theory / Supercongruences for truncated hypergeometric series',
  'Combinatorics / Hypergeometric identities and multinomial coefficients',
  'Algebraic Number Theory / Binary quadratic forms and representations of primes',
  'Modular Forms / CM forms and Hecke eigenvalues over quadratic fields'],
 'keywords': ['Legendre symbol',
  'central multinomial coefficient',
  'truncated hypergeometric series (4F3 at 1)',
  'finite field hypergeometric functions (Greene/McCarthy)',
  'p-adic gamma function',
  'Gross–Koblitz formula',
  'Gauss sums and Hasse–Davenport relation',
  'binary quadratic forms x^2 + Dy^2',
  'supercongruence modulo p^2 and p^3',
  'Wilf–Zeilberger (WZ) method'],
 'key_ideas': ['hypergeometric reparameterization of factorial ratios via Pochhammer symbols',
  'translation of truncated sums to finite field hypergeometric values',
  'case analysis controlled by two independent quadratic characters and their product',
  'evaluation through p

## Step 3 — Metadata → New graduate-level problem

In [14]:
GEN_PROMPT = """You are a problem composer designing a NEW graduate-level mathematics problem from a metadata blueprint.

Constraints:
- The problem must be self-contained: define every symbol and assumption.
- It must genuinely require the listed key ideas and techniques — not name them, but force the solver to use them.
- Difficulty: graduate level. Non-trivial, but solvable in 1-2 pages by an expert in the listed prerequisites.
- Do NOT mention the metadata or your reasoning. Output the problem statement only.
- Prefer a clean ask ("prove", "compute", "classify", "determine whether").
- Do not include additional instructions.
- The question should require one-single short-form answer as final output.

Metadata blueprint:
{blueprint}

Write the problem now.
"""

def generate_problem(blueprint: dict) -> str:
    resp = litellm.completion(
        model=MODEL,
        messages=[{"role": "user", "content": GEN_PROMPT.replace("{blueprint}", json.dumps(blueprint, indent=2))}],
    )
    return resp.choices[0].message.content.strip()

In [ ]:
generated = []
for rec in metadata_records:
    bp = sample_metadata(rec["metadata"], seed=hash(rec["question_id"]) & 0xFFFFFFFF)
    new_problem = generate_problem(bp)
    generated.append({
        "question_id": rec["question_id"],
        "paper_id": rec.get("paper_id"),
        "blueprint": bp,
        "new_problem": new_problem,
    })
    print("="*80)
    print(rec["question_id"], "blueprint:", bp)
    print("-"*80)
    print(new_problem)

with open(HERE / "generated.json", "w") as f:
    json.dump(generated, f, indent=2, ensure_ascii=False)

q_018 blueprint: {'categories': ['Modular Forms / CM forms and Hecke eigenvalues over quadratic fields', 'Combinatorics / Hypergeometric identities and multinomial coefficients'], 'keywords': ['p-adic gamma function', 'finite field hypergeometric functions (Greene/McCarthy)', 'Gross–Koblitz formula', 'supercongruence modulo p^2 and p^3'], 'key_ideas': ['translation of truncated sums to finite field hypergeometric values', 'evaluation through p-adic gamma expansions to second and third order', 'case analysis controlled by two independent quadratic characters and their product'], 'prerequisites': ['p-adic valuations; Lucas and Kummer congruences; lifting-the-exponent', 'generalized hypergeometric series and Pochhammer notation'], 'techniques': ['genus theory to characterize splitting/representation by quadratic forms', 'Greene–McCarthy finite field hypergeometric evaluation with Gross–Koblitz'], 'difficulty': 'research'}
-------------------------------------------------------------------